<a href="https://colab.research.google.com/github/Thangamuniyandi894/mini_project_3/blob/main/job_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df=pd.read_csv("/content/HR_Job_Placement_Dataset.csv")

## **understanding the data**

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
(df.isnull().sum()/len(df))*100

In [ ]:
df.duplicated().sum()

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.describe()

In [ ]:
cat_col=df.select_dtypes(include="object").columns
for col in cat_col:
  print(f"____{col}_____")
  print(df[col].unique())
  print()

In [ ]:
for col in cat_col:
  print(f"____{col}_____")
  print(df[col].value_counts())
  print()

In [ ]:
(df[cat_col].isnull().sum()/len(df))*100

In [ ]:
num_col=df.select_dtypes(include="number").columns
print(num_col)
(df[num_col].isnull().sum()/len(df))*100

# **data preprocessing**

In [ ]:
df=df.copy()

In [ ]:
for col in cat_col:
    df[col] = df[col].str.title()

In [ ]:
for col in cat_col:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [ ]:
for col in cat_col:
  print(f"____{col}_____")
  print(df[col].unique())
  print()

In [ ]:
for col in cat_col:
    df[col] = df[col].fillna('Unknown')

In [ ]:
for col in num_col:
    df[col] = df[col].fillna(df[col].median())

# **Outlier handling**

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns


for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col} - Outliers count: {len(outliers)}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols, 1):
    plt.subplot(4, 4, i)
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

**Add derived columns**

In [ ]:
# 1. Experience category (Fresher / Junior / Senior)
def get_exp_cat(exp):
    if exp == 0:
        return 'Fresher'
    elif exp <= 3:
        return 'Junior'
    else:
        return 'Senior'

df['experience_category'] = df['years_of_experience'].apply(get_exp_cat)

In [ ]:
# 2. Academic performance bands
avg_score = (df['ssc_percentage'] + df['hsc_percentage'] + df['degree_percentage']) / 3

conditions_acad = [
    (avg_score >= 75),
    (avg_score >= 60) & (avg_score < 75),
    (avg_score < 60)
]
choices_acad = ['First Class Distinction', 'First Class', 'Second Class']
df['academic_performance_band'] = np.select(conditions_acad, choices_acad, default='Average')

In [ ]:
# 3. Skills match level (Low / Medium / High)
conditions_skill = [
    (df['skills_match_percentage'] >= 75),
    (df['skills_match_percentage'] >= 50) & (df['skills_match_percentage'] < 75),
    (df['skills_match_percentage'] < 50)
]
choices_skill = ['High', 'Medium', 'Low']
df['skills_match_level'] = np.select(conditions_skill, choices_skill, default='Low')

In [ ]:
# 4. Interview performance category
interview_avg = (df['technical_score'] + df['aptitude_score'] + df['communication_score']) / 3

conditions_interview = [
    (interview_avg >= 75),
    (interview_avg >= 50) & (interview_avg < 75),
    (interview_avg < 50)
]
choices_interview = ['Excellent', 'Good', 'Average']
df['interview_performance_category'] = np.select(conditions_interview, choices_interview, default='Average')

In [ ]:
# 5. Placement probability score
df['placement_probability_score'] = (
    (df['skills_match_percentage'] * 0.4) +
    (interview_avg * 0.4) +
    (avg_score * 0.2)
).round(2)

In [ ]:
df

# **EDA analysis**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='skills_match_percentage', hue='status', common_norm=False, fill=True)
plt.title('Skills Match Percentage Impact on Placement')
plt.xlabel('Skills Match Percentage')
plt.ylabel('Density')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='company_tier', hue='status', data=df, palette='Set2')
plt.title('Company Tier vs Job Acceptance Rate')
plt.xlabel('Company Tier')
plt.ylabel('Number of Students')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(x='status', y='years_of_experience', data=df, ci=None, palette='Set2')

plt.title('Years of Experience vs Placement Probability')
plt.xlabel('Placement Status')
plt.ylabel('Average Years of Experience')

for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.1f} Years",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 8), textcoords='offset points')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='competition_level', hue='status', data=df, palette='Set2')
plt.title('Competition Level vs Job Acceptance')
plt.xlabel('Competition Level')
plt.ylabel('Number of Students')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Analysis Among Numeric Features')
plt.show()

In [ ]:
df.shape

In [ ]:
df.to_csv('Hr_job_analysis.csv',index=False)

In [ ]:
df.shape